# Analysis after the Sentiment Analysis

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

input_file = Path("../data/sentiment_results/all_companies_sentiment.csv")
output_folder = Path("../data/sentiment_results/graphs")
output_folder.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(input_file)

required = {
    "ID", "Company", "Country", "Year", "Communication_type",
    "Leadership_context", "Information_valence",
    "Sentiment_label", "Sentiment_score",
    "P_positive", "P_neutral", "P_negative",
}

missing = required - set(data.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

ids = data["ID"].astype("string").str.strip()
if ids.isna().any() or ids.eq("").any() or ids.duplicated().any():
    raise ValueError("Review missing or duplicate IDs before analysis.")

# Normalize categories
data["Company"] = data["Company"].astype("string").str.strip().str.lower()
data["Country"] = data["Country"].astype("string").str.strip().str.upper()
data["Leadership_context"] = (
    data["Leadership_context"].astype("string").str.strip().str.upper()
)
data["Sentiment_label"] = (
    data["Sentiment_label"].astype("string").str.strip().str.lower()
)

valence = (
    data["Information_valence"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "", regex=True)
)

data["Valence_group"] = valence.map({
    "positive": "Positive",
    "neutral/mixed": "Neutral/Mixed",
    "negative": "Negative",
})

numeric_columns = [
    "Year", "Sentiment_score",
    "P_positive", "P_neutral", "P_negative",
]

for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors="coerce")

probabilities = data[["P_positive", "P_neutral", "P_negative"]]

valid_scores = (
    probabilities.notna().all(axis=1)
    & probabilities.ge(0).all(axis=1)
    & probabilities.le(1).all(axis=1)
    & np.isclose(probabilities.sum(axis=1), 1, atol=1e-4)
    & data["Sentiment_score"].between(-1, 1)
    & np.isclose(
        data["Sentiment_score"],
        data["P_positive"] - data["P_negative"],
        atol=1e-4,
    )
    & data["Sentiment_label"].isin(["negative", "neutral", "positive"])
)

communication = (
    data["Communication_type"].astype("string").str.strip().str.lower()
)

# Rebuild the flag rather than relying on the saved True/False values
data["CEO_analysis"] = (
    communication.isin(
        ["interview_quote", "interview", "message", "letter"]
    )
    & valid_scores
)

ceo = data.loc[data["CEO_analysis"]].copy()

if ceo.empty:
    raise ValueError("No valid CEO observations found. Check types and scores.")

print(f"All rows: {len(data)}")
print(f"CEO observations included: {len(ceo)}")
print(f"Rows excluded: {len(data) - len(ceo)}")
print(f"CEO rows without recognized valence: {ceo['Valence_group'].isna().sum()}")

display(
    ceo.groupby("Company").agg(
        observations=("ID", "size"),
        years=("Year", "nunique"),
        mean_sentiment=("Sentiment_score", "mean"),
        median_sentiment=("Sentiment_score", "median"),
    ).round(3)
)

sns.set_theme(style="whitegrid")

sentiment_order = ["negative", "neutral", "positive"]
valence_order = ["Negative", "Neutral/Mixed", "Positive"]
context_order = ["PERF", "STRAT", "MARKET", "CHALL", "PEOPLE", "STAKE"]

def save_plot(fig, filename):
    fig.tight_layout()
    fig.savefig(
        output_folder / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

In [ ]:
comparison = ceo.dropna(subset=["Valence_group"])

if comparison.empty:
    raise ValueError("No recognized Information_valence values to compare.")

counts = pd.crosstab(
    comparison["Valence_group"],
    comparison["Sentiment_label"],
).reindex(
    index=valence_order,
    columns=sentiment_order,
    fill_value=0,
)

percentages = (
    counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0) * 100
)

annotations = pd.DataFrame(
    "",
    index=counts.index,
    columns=counts.columns,
)

for row in counts.index:
    for column in counts.columns:
        value = percentages.loc[row, column]
        if pd.notna(value):
            annotations.loc[row, column] = (
                f"{value:.1f}%\n(n={counts.loc[row, column]})"
            )

fig, ax = plt.subplots(figsize=(9, 6))

sns.heatmap(
    percentages,
    annot=annotations,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "Percentage within information-valence category"},
    ax=ax,
)

ax.set_title("Information valence and BERT sentiment")
ax.set_xlabel("BERT sentiment label")
ax.set_ylabel("Manually coded information valence")

save_plot(fig, "01_valence_vs_sentiment")

counts.to_csv(output_folder / "valence_sentiment_counts.csv")
percentages.to_csv(output_folder / "valence_sentiment_percentages.csv")

In [ ]:
comparisons = [
    (
        "Valence_group",
        valence_order,
        "Sentiment by information valence",
        "02_sentiment_by_valence",
    ),
    (
        "Leadership_context",
        context_order,
        "Sentiment by leadership context",
        "03_sentiment_by_context",
    ),
    (
        "Company",
        sorted(ceo["Company"].dropna().unique()),
        "Sentiment by company",
        "04_sentiment_by_company",
    ),
]

for column, requested_order, title, filename in comparisons:
    plot_data = ceo.dropna(subset=[column])
    order = [
        value for value in requested_order
        if value in plot_data[column].unique()
    ]

    if not order:
        continue

    fig, ax = plt.subplots(figsize=(11, 6))

    sns.boxplot(
        data=plot_data,
        x=column,
        y="Sentiment_score",
        order=order,
        color="lightsteelblue",
        showfliers=False,
        ax=ax,
    )

    sns.stripplot(
        data=plot_data,
        x=column,
        y="Sentiment_score",
        order=order,
        color="black",
        alpha=0.45,
        jitter=0.15,
        size=4,
        ax=ax,
    )

    sample_sizes = plot_data[column].value_counts()
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([
        f"{value}\n(n={sample_sizes[value]})"
        for value in order
    ])

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_ylim(-1.05, 1.05)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Sentiment score: P(positive) − P(negative)")

    save_plot(fig, filename)

In [ ]:
context_country = ceo.dropna(
    subset=["Leadership_context", "Country"]
)

means = context_country.pivot_table(
    index="Leadership_context",
    columns="Country",
    values="Sentiment_score",
    aggfunc="mean",
)

sample_sizes = pd.crosstab(
    context_country["Leadership_context"],
    context_country["Country"],
)

available_contexts = [
    context for context in context_order
    if context in means.index
]

means = means.reindex(available_contexts)
sample_sizes = sample_sizes.reindex(
    index=means.index,
    columns=means.columns,
    fill_value=0,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

sns.heatmap(
    means,
    annot=True,
    fmt=".2f",
    cmap="RdBu",
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[0],
)
axes[0].set_title("Mean sentiment score")

sns.heatmap(
    sample_sizes,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=axes[1],
)
axes[1].set_title("Number of CEO observations")

for ax in axes:
    ax.set_xlabel("Country")
    ax.set_ylabel("Leadership context")

save_plot(fig, "05_context_by_country")

In [ ]:
# Select rows where the two labels differ after aligning category names.
# This is a comparison of different constructs, not an accuracy test.

valence_reference = ceo["Valence_group"].map({
    "Positive": "positive",
    "Neutral/Mixed": "neutral",
    "Negative": "negative",
})

interesting = ceo.loc[
    valence_reference.notna()
    & ceo["Sentiment_label"].notna()
    & valence_reference.ne(ceo["Sentiment_label"])
].copy()

interesting["Comparison"] = (
    interesting["Valence_group"]
    + " information / "
    + interesting["Sentiment_label"]
    + " sentiment"
)

review_columns = [
    "ID",
    "Company",
    "Leadership_context",
    "Information_valence",
    "Sentiment_label",
    "Sentiment_score",
    "Comparison",
    "Text",
]

print(f"Passages selected for review: {len(interesting)}")

display(interesting[review_columns])

interesting[review_columns].to_csv(
    output_folder / "passages_for_qualitative_review.csv",
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
interesting = ceo.loc[
    (
        ceo["Valence_group"].eq("Negative")
        & ceo["Sentiment_label"].eq("positive")
    )
    |
    (
        ceo["Valence_group"].eq("Positive")
        & ceo["Sentiment_label"].eq("negative")
    )
].copy()

review_columns = [
    "ID",
    "Company",
    "Leadership_context",
    "Information_valence",
    "Sentiment_label",
    "Sentiment_score",
    "Text",
]

display(interesting[review_columns])

interesting[review_columns].to_csv(
    output_folder / "passages_for_qualitative_review.csv",
    index=False,
    encoding="utf-8-sig",
)